In [ ]:
%matplotlib widget
import os
import sys
import time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torchkbnufft as tkbn
import pypulseq as pp
import MRzeroCore as mr0
import matplotlib.pyplot as plt

def _find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for d in [cur, *cur.parents]:
        if (d / "Modules").is_dir():
            return d
    raise RuntimeError(f"Could not find a folder containing 'Modules' from {cur}")

REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Drop any cached Modules* modules so a kernel restart is only needed for
# *new* modules, not for edits to existing ones - re-running this cell
# always re-imports the current Modules/ code.
for _m in [m for m in list(sys.modules) if m == 'Modules' or m.startswith('Modules.')]:
    del sys.modules[_m]

from Modules import PARAMS, build_sim_data
from Modules.bssfp_params import TISSUES, T2STAR, NAMES
from Modules.fast_iso import isochromat_sim_fast  

out_dir  = REPO_ROOT / "Simulation & Visualization" / "Signals"   # <-- signal .pt files land here
signal_name = "bssfp_kooshball3D_signal.pt"   # <-- Change signal file name
os.makedirs(out_dir, exist_ok=True)

torch.set_default_device("cuda")
device = torch.device("cuda")

In [ ]:
# locate the .seq file 
# A .seq is a pypulseq sequence file, NOT a Python module, so it cannot be imported. Point at it on disk and load it with mr0.Sequence.import_file.

seq_name = "bssfp_kooshball3D_signal_n64_R384_S25.seq"   # <-- choose sequence
seq_file = REPO_ROOT / "Sequences" / "Sequence files" / seq_name
if not seq_file.exists():
    raise FileNotFoundError(
        f"Missing .seq: {seq_file}\nRun 3D_bssfp_sequence.ipynb first to generate it.")
print("sequence file:", seq_file)

In [ ]:
mr_seq = mr0.Sequence.import_file(str(seq_file))   # <-- import the sequence
print("imported sequence, repetitions:", len(mr_seq))

In [ ]:
# ---- simulation data (phantom + coils) via the SAME shared pipeline ---------
# used by 3D_bssfp_sequence.ipynb, so the simulated signal and the sequence are
# guaranteed to share the same grid / coil setup.  Builds on CPU (fast, no
# VRAM needed for setup); the sim cell moves it to GPU with data.cuda().
data, masks, brain_mask, coil_den, coil_c64 = build_sim_data(
    raw_path=str(REPO_ROOT / "Data" / "subject04_crisp_v.rawb"),  # <-- <repo>/Data/
    p=PARAMS,
    device=torch.device("cpu"),
    permute_order=(2, 1, 0),
)

coil_den = coil_den.to(device)
coil_c64 = coil_c64.to(device)
# Bind the grid / readout names the cells below already use (same values as the
# sequence notebook, straight from the shared protocol params):
n          = PARAMS.n
FOV        = PARAMS.FOV
N_x        = PARAMS.N_x
N_readout  = PARAMS.N_readout
N_segments = PARAMS.N_segments
N_channels = PARAMS.N_channels
n_adc      = PARAMS.n_adc
TR_echo    = PARAMS.TR_echo

In [ ]:
signal = isochromat_sim_fast(mr_seq, data.cuda(), spin_count=64, spin_dist="rand", print_progress=True, chunk_bytes=2**30)
signal = signal.to(device).permute((1, 0)).contiguous().to(torch.complex64)

signal_path = os.path.join(out_dir, signal_name)
torch.save(signal, signal_path) 
assert tuple(signal.shape) == (PARAMS.N_channels, PARAMS.n_adc), (tuple(signal.shape), (PARAMS.N_channels, PARAMS.n_adc))
print("simulated signal:", tuple(signal.shape))

***Plot the the the signal evolution and NUFFT preview***

In [ ]:
def kspace_to_rad(mr_seq, n, FOV):
    'ADC k-space (1/m) -> rad/pixel of an n^3 grid spanning FOV.  Returns (3, N_adc).'
    k = torch.as_tensor(mr_seq.get_kspace()).float()
    k = k.reshape(-1, k.shape[-1])[:, :3].T.contiguous()
    return k * (2 * np.pi * FOV / n)

k_full = kspace_to_rad(mr_seq, n, FOV)
assert k_full.shape[1] == signal.shape[1], (k_full.shape, signal.shape)
k_full = k_full.to(device)

# samples with |k| > pi lie beyond the grid's kmax and must be DROPPED (not
# clamped) for the gridding recon; CG uses the full set with k normalized to
# [-0.5, 0.5] cycles/voxel.
keep_full = k_full.norm(dim=0) <= np.pi
k_norm = k_full / k_full.norm(dim=0).max() * 0.5     # cycles/voxel, for the CG operator
print(f"keep {int(keep_full.sum())}/{keep_full.numel()} samples (|k| <= pi)")

In [ ]:
def nufft_recon(sig, k_rad, n):
    'Per-channel density-compensated adjoint NUFFT.  sig: (C, N), k_rad: (3, N) -> (C, n, n, n).'
    keep = k_rad.norm(dim=0) <= np.pi
    k, s = k_rad[:, keep], sig[:, keep]
    dcomp = k.norm(dim=0).clamp(min=np.pi / n) ** 2   # |k|^2 for radial
    adj = tkbn.KbNufftAdjoint(im_size=(n, n, n), grid_size=(2 * n,) * 3).to(sig.device)
    return adj((s * dcomp)[None], k)[0]

def combine(img_c):
    'adaptive SENSE combine of a (C, n, n, n) multi-coil image -> (n, n, n) complex'
    return (coil_c64.conj() * img_c).sum(0) / coil_den.clamp(min=1e-2 * coil_den.max())

def frame_spoke_blocks(p):
    'ADC block (N_x samples) index of echo p in every segment'
    return [s * N_readout + p for s in range(N_segments)]

def frame_idx(p):
    return torch.cat([torch.arange(b * N_x, (b + 1) * N_x) for b in frame_spoke_blocks(p)])

ti_effs = PARAMS.effective_tis(TR_echo)
t1 = time.time()
preview = {}
for p in range(0, N_readout, 24):        # 16 frames
    idx = frame_idx(p)
    preview[p] = combine(nufft_recon(signal[:, idx], k_full[:, idx], n))
print(f"preview recon ({len(preview)} frames): {time.time()-t1:.0f} s")

# signed (steady-state phase-referenced) tissue-mean transient
ref_phase = torch.angle(preview[max(preview)])
tvals = np.array([ti_effs[p] for p in sorted(preview)])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
mid = n // 2
ax[0].imshow(np.rot90(preview[168][:, :, mid].T.abs().cpu().numpy(), 2), cmap="gray")
ax[0].set_title(f"NUFFT preview, frame 168\nTI = {ti_effs[168]*1e3:.0f} ms")
ax[0].axis("off")
for t in NAMES:
    m = torch.as_tensor(masks[t], device=device)
    s = [float(torch.real(preview[p] * torch.exp(-1j * ref_phase))[m].mean())
         for p in sorted(preview)]
    ax[1].plot(tvals * 1e3, s, label=t)
ax[1].axhline(0, color="k", lw=0.5)
ax[1].set_title("signed tissue-mean transient (SS-phase referenced)")
ax[1].set_xlabel("TI (ms)"); ax[1].set_ylabel("signal"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

***Plot the fibonacci sphere showing spokes***

In [ ]:
K3 = k_full.reshape(3, N_segments, N_readout, N_x).cpu().numpy()   # (3, S, P, J)
r  = np.linalg.norm(K3, axis=0)                                    # (S, P, J)
j_edge = r.argmax(axis=-1)                                         # outermost sample per spoke
si, pi = np.meshgrid(np.arange(N_segments), np.arange(N_readout), indexing='ij')
tips = K3[:, si.ravel(), pi.ravel(), j_edge.ravel()]               # (3, S*P)
endpoints = tips / np.linalg.norm(tips, axis=0, keepdims=True)     # onto unit sphere
order = np.arange(endpoints.shape[1])   # temporal = golden-angle spoke order


fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(endpoints[0], endpoints[1], endpoints[2], c=order, cmap='viridis', s=1.5, linewidths=0)
fig.colorbar(sc, ax=ax, shrink=0.6, label='spoke (echo) order')
ax.set_box_aspect((1, 1, 1))
ax.set_xlabel('kx'); ax.set_ylabel('ky'); ax.set_zlabel('kz')
ax.set_title(f'fib-sphere trajectory - {endpoints.shape[0]} spoke endpoints (no spokes)')
plt.show()